In [45]:
# 01 – Data preprocessing & differential expression (GSE266618)

#Paper:
#Datlinger P et al. *Systematic discovery of CRISPR-boosted CAR T cell immunotherapies.*
#Nature 2025;646(8086):963–972.
#DOI: 10.1038/s41586-025-9507-y

#Dataset:
#- Bulk RNA-seq counts: `GSE266618_counts.csv.gz` (Supplementary file)
#- GEO series matrix (for sample metadata): `GSE266618_series_matrix.txt` 


In [46]:
from pathlib import Path
import os

# Set repo root relative to current working directory
REPO_ROOT = Path.cwd().parents[0]  # goes up from notebooks/ → repo root
os.chdir(REPO_ROOT)

print("Working directory set to:", Path.cwd())

Working directory set to: /Users/acastano/Desktop


In [47]:
import os

os.chdir("/Users/acastano/Desktop/crispr_carT_AI_analysis")
print("CWD now:", os.getcwd())
!ls


CWD now: /Users/acastano/Desktop/crispr_carT_AI_analysis
~$per in progress-RHOG modulates CAR-T activation trajectories without inducing new transcriptional states.docx
analysis_utils
archive
config.json
data
demo.txt
environment.yml
figures
jupyter.log
manuscript
metadata_with_modules_and_trajectories.csv
notebooks
PROJECT_MEMORY.md
README.md
results
scripts
zshrc_backup.txt


In [48]:
import pandas as pd

expr_counts = pd.read_csv("data/raw/GSE266618_counts.csv", index_col=0)
expr_counts.shape
expr_counts.head()

FileNotFoundError: [Errno 2] No such file or directory: 'data/raw/GSE266618_counts.csv'

In [28]:
expr_counts.shape


NameError: name 'expr_counts' is not defined

In [29]:
expr_counts.index[:10]
expr_counts.columns[:10]


NameError: name 'expr_counts' is not defined

In [30]:
#Normalize Expression
import numpy as np
lib_sizes = expr_counts.sum(axis=0)
cpm = expr_counts.divide(lib_sizes, axis=1) * 1e6
expr_logcpm = np.log2(cpm + 1)

expr_logcpm.shape
expr_logcpm.iloc[:5, :5]


NameError: name 'expr_counts' is not defined

In [31]:
#Parse metadata from the series matrix
series_path = "data/raw/GSE266618_series_matrix.txt"


In [32]:
#Extract the “title” and “geo_accession” fields:
sample_meta_raw = {}

with open(series_path, "r") as f:
    for line in f:
        line = line.strip()
        if line.startswith("!Sample_"):
            parts = line.split("\t")
            key = parts[0].replace("!Sample_", "")  # e.g. "title", "geo_accession"
            values = parts[1:]
            sample_meta_raw[key] = values
        elif line.startswith("!series_matrix_table_begin"):
            break

metadata = pd.DataFrame(index=sample_meta_raw["geo_accession"])
metadata["title"] = sample_meta_raw["title"]
metadata.head()


FileNotFoundError: [Errno 2] No such file or directory: 'data/raw/GSE266618_series_matrix.txt'

In [33]:
#Extract biological fields from the title

def parse_title(title):
    parts = [p.strip() for p in title.split(",")]
    cell_type, time_str, guide, donor = parts
    hours = int(time_str.replace("h", "").strip())
    return pd.Series({
        "cell_type": cell_type,
        "hours": hours,
        "guide": guide,
        "donor": donor
    })

parsed = metadata["title"].apply(parse_title)
metadata = pd.concat([metadata, parsed], axis=1)
metadata


NameError: name 'metadata' is not defined

In [34]:
import os
os.getcwd()



'/Users/acastano/Desktop/crispr_carT_AI_analysis'

In [35]:
os.chdir("/Users/acastano/Desktop/crispr_carT_AI_analysis")
os.getcwd()


'/Users/acastano/Desktop/crispr_carT_AI_analysis'

In [36]:
!ls data/raw


ls: data/raw: No such file or directory


In [37]:
import pandas as pd

series_path = "data/raw/GSE266618_series_matrix.txt"

sample_meta_raw = {}

with open(series_path, "r") as f:
    for line in f:
        line = line.strip()
        if line.startswith("!Sample_"):
            parts = line.split("\t")
            key = parts[0].replace("!Sample_", "")
            values = parts[1:]
            sample_meta_raw[key] = values
        elif line.startswith("!series_matrix_table_begin"):
            break

print("Fields available:", list(sample_meta_raw.keys()))


FileNotFoundError: [Errno 2] No such file or directory: 'data/raw/GSE266618_series_matrix.txt'

In [38]:
list(sample_meta_raw.keys())
sample_meta_raw["title"][:5]


KeyError: 'title'

In [39]:
#Rebuild sample_meta_raw
series_path = "data/raw/GSE266618_series_matrix.txt"

sample_meta_raw = {}
with open(series_path, "r") as f:
    for line in f:
        line = line.strip()
        if line.startswith("!Sample_"):
            parts = line.split("\t")
            key = parts[0].replace("!Sample_", "")
            values = parts[1:]
            sample_meta_raw[key] = values
        elif line.startswith("!series_matrix_table_begin"):
            break

list(sample_meta_raw.keys())


FileNotFoundError: [Errno 2] No such file or directory: 'data/raw/GSE266618_series_matrix.txt'

In [40]:
import os
os.chdir("/Users/acastano/Desktop/crispr_carT_AI_analysis")
print(os.getcwd())
!ls data/raw


/Users/acastano/Desktop/crispr_carT_AI_analysis
ls: data/raw: No such file or directory


In [41]:
import pandas as pd

expr_counts = pd.read_csv(
    "data/raw/GSE266618_counts.csv",
    index_col=0
)

expr_counts.shape, expr_counts.head()


FileNotFoundError: [Errno 2] No such file or directory: 'data/raw/GSE266618_counts.csv'

In [42]:
sample_meta_raw = {}

with open("data/raw/GSE266618_series_matrix.txt", "r") as f:
    for line in f:
        line = line.strip()
        if line.startswith("!Sample_"):
            parts = line.split("\t")
            key = parts[0].replace("!Sample_", "")
            values = parts[1:]
            sample_meta_raw[key] = values
        elif line.startswith("!series_matrix_table_begin"):
            break

sample_meta_raw.keys()



FileNotFoundError: [Errno 2] No such file or directory: 'data/raw/GSE266618_series_matrix.txt'

In [43]:
n_samples_counts = expr_counts.shape[1]
titles_raw = pd.Series(sample_meta_raw["title"], name="title")
gsm_ids    = pd.Series(sample_meta_raw["geo_accession"], name="GSM")

print("Counts:", n_samples_counts, "Titles:", len(titles_raw), "GSMs:", len(gsm_ids))


NameError: name 'expr_counts' is not defined

In [44]:
metadata = pd.DataFrame({
    "sample_id": expr_counts.columns,
    "GSM": gsm_ids.values,
    "title": titles_raw.values
}).set_index("sample_id")

metadata.head()


NameError: name 'expr_counts' is not defined

In [82]:
def parse_title(title):
    # Remove any surrounding quotes
    title = title.strip('"')
    parts = [p.strip() for p in title.split(",")]
    
    # Expect: cell_type, time, guide, donor
    cell_type, time_str, guide, donor = parts
    hours = int(time_str.replace("h", "").strip())
    
    return pd.Series({
        "cell_type": cell_type,
        "hours": hours,
        "guide": guide,
        "donor": donor
    })

parsed = metadata["title"].apply(parse_title)
metadata = pd.concat([metadata, parsed], axis=1)

metadata.head()


,GSM,title,cell_type,hours,guide,donor
sample_id,,,,,,
CART0077_RNAseq_NgsRun1_001,"""GSM8252546""","""CD4, 0h, SafeHarbor, CART0077_D1""",CD4,0,SafeHarbor,CART0077_D1
CART0077_RNAseq_NgsRun1_002,"""GSM8252547""","""CD4, 0h, RHOG, CART0077_D1""",CD4,0,RHOG,CART0077_D1
CART0077_RNAseq_NgsRun1_003,"""GSM8252548""","""CD4, 0h, SafeHarbor, CART0077_D2""",CD4,0,SafeHarbor,CART0077_D2
CART0077_RNAseq_NgsRun1_004,"""GSM8252549""","""CD4, 0h, RHOG, CART0077_D2""",CD4,0,RHOG,CART0077_D2
CART0077_RNAseq_NgsRun1_005,"""GSM8252550""","""CD4, 0h, SafeHarbor, CART0077_D3""",CD4,0,SafeHarbor,CART0077_D3


In [83]:
metadata["hours"].value_counts().sort_index()


hours
0      12
24     12
72     12
168    12
240    12
Name: count, dtype: int64

In [84]:
metadata["guide"].value_counts()


guide
SafeHarbor    30
RHOG          30
Name: count, dtype: int64

In [85]:
metadata["donor"].value_counts()


donor
CART0077_D1    20
CART0077_D2    20
CART0077_D3    20
Name: count, dtype: int64

In [86]:
import numpy as np

lib_sizes = expr_counts.sum(axis=0)
cpm = expr_counts.divide(lib_sizes, axis=1) * 1e6
expr_logcpm = np.log2(cpm + 1)

expr_logcpm.shape


(60675, 60)

In [87]:
from scipy import stats
from statsmodels.stats.multitest import multipletests

def differential_expression(expr_df, metadata, mask_group1, mask_group2,
                            group1_name="group1", group2_name="group2"):
    samples1 = metadata.index[mask_group1]
    samples2 = metadata.index[mask_group2]
    
    X1 = expr_df.loc[:, samples1]
    X2 = expr_df.loc[:, samples2]
    
    mean1 = X1.mean(axis=1)
    mean2 = X2.mean(axis=1)
    log2_fc = mean2 - mean1
    
    # Welch’s t-test
    t_stats, pvals = stats.ttest_ind(X2.T, X1.T, equal_var=False, nan_policy="omit")
    
    # FDR correction
    _, pval_adj, _, _ = multipletests(pvals, method="fdr_bh")
    
    return pd.DataFrame({
        f"mean_{group1_name}": mean1,
        f"mean_{group2_name}": mean2,
        "log2_fc": log2_fc,
        "pval": pvals,
        "pval_adj": pval_adj
    })


In [88]:
early_mask = metadata["hours"] == 0
late_mask  = metadata["hours"] >= 168

de_late_vs_early = differential_expression(
    expr_logcpm, metadata,
    mask_group1=early_mask,
    mask_group2=late_mask,
    group1_name="early_0h",
    group2_name="late_168hplus"
)

de_late_vs_early.head()


,mean_early_0h,mean_late_168hplus,log2_fc,pval,pval_adj
gene,,,,,
ENSG00000223972,0.055567,0.049877,-0.005689,0.811482,NaN
ENSG00000227232,2.649494,2.853152,0.203658,0.187170,NaN
ENSG00000278267,0.242513,0.366924,0.124411,0.025860,NaN
ENSG00000243485,0.018491,0.013329,-0.005163,0.801665,NaN
ENSG00000284332,0.000000,0.000000,0.000000,NaN,NaN


In [89]:
metadata.columns

Index(['GSM', 'title', 'cell_type', 'hours', 'guide', 'donor'], dtype='object')

In [90]:
expr_symbol.shape

NameError: name 'expr_symbol' is not defined

In [91]:
(n_genes, n_samples)

NameError: name 'n_genes' is not defined